# vLLM Inference Benchmark

## Objective

Measure the performance impact of `max-num-seqs` on a single-GPU
vLLM inference server.

### Experimental variable

- `max-num-seqs`: 4, 8, 12, 16

### Fixed variables

- Model: Qwen/Qwen2.5-1.5B-Instruct
- GPU: NVIDIA Tesla T4
- GPU count: 1
- max-model-len: 2000
- gpu-memory-utilization: 0.85
- tensor-parallel-size: 1
- pipeline-parallel-size: 1
- Input length: 430 tokens
- Output length: 1000 tokens
- Request rate: 2 RPS
- Requests per experiment: 50


## 1. Imports and Configuration

In [ ]:
import os
import sys
import time
import json
import signal
import subprocess
from pathlib import Path

import requests
import pandas as pd
import matplotlib.pyplot as plt

print("Python:", sys.version)


In [ ]:
# ------------------------------------------------------------
# Benchmark configuration
# ------------------------------------------------------------

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SERVED_NAME = "llm-model"

HOST = "127.0.0.1"
PORT = 8000
BASE_URL = f"http://{HOST}:{PORT}"

MAX_MODEL_LEN = 2000
GPU_MEMORY_UTILIZATION = 0.85

TENSOR_PARALLEL_SIZE = 1
PIPELINE_PARALLEL_SIZE = 1

# Experimental variable
MAX_NUM_SEQS_VALUES = [4, 8, 12, 16]

# Workload
NUM_PROMPTS = 50
REQUEST_RATE = 2

INPUT_LEN = 430
OUTPUT_LEN = 1000

# Result directories
RESULT_DIR = Path("results/raw")
PLOT_DIR = Path("plots")

RESULT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded")
print("Model:", MODEL)
print("GPU memory utilization:", GPU_MEMORY_UTILIZATION)
print("max-num-seqs values:", MAX_NUM_SEQS_VALUES)
print("Requests:", NUM_PROMPTS)
print("Request rate:", REQUEST_RATE, "RPS")


## 2. GPU Verification

In [ ]:
result = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("nvidia-smi failed")


## 3. vLLM Verification

In [ ]:
# Install the pinned benchmark version if it is not already available.
# Kaggle may already have vLLM installed.

import importlib.util

if importlib.util.find_spec("vllm") is None:
    print("vLLM not found. Installing vLLM 0.30.0...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-U", "vllm==0.30.0"],
        check=True
    )
else:
    print("vLLM package already installed.")


In [ ]:
import vllm

print("vLLM version:", vllm.__version__)

if vllm.__version__ != "0.30.0":
    print("WARNING: expected vLLM 0.30.0")


## 4. Start vLLM Server

In [ ]:
server_process = None

def start_vllm(max_num_seqs):
    global server_process

    if server_process is not None and server_process.poll() is None:
        raise RuntimeError("vLLM server is already running.")

    command = [
        "vllm",
        "serve",
        MODEL,
        "--served-model-name", SERVED_NAME,
        "--host", "0.0.0.0",
        "--port", str(PORT),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--max-num-seqs", str(max_num_seqs),
        "--tensor-parallel-size", str(TENSOR_PARALLEL_SIZE),
        "--pipeline-parallel-size", str(PIPELINE_PARALLEL_SIZE),
        "--dtype", "half",
    ]

    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0"

    log_file = open("vllm_server.log", "w")

    print("Starting vLLM:")
    print(" ".join(command))

    server_process = subprocess.Popen(
        command,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=env,
    )

    return server_process


## 5. Wait for Server

In [ ]:
def wait_for_server(timeout=300):
    health_url = f"{BASE_URL}/health"

    start_time = time.time()

    while time.time() - start_time < timeout:

        if server_process is not None:
            if server_process.poll() is not None:
                raise RuntimeError(
                    "vLLM server exited before becoming healthy. "
                    "Check vllm_server.log"
                )

        try:
            response = requests.get(
                health_url,
                timeout=5
            )

            if response.status_code == 200:
                print("vLLM server is healthy.")
                return True

        except requests.RequestException:
            pass

        time.sleep(2)

    raise TimeoutError(
        f"vLLM server did not become healthy within {timeout} seconds."
    )


In [ ]:
def get_model_info():
    url = f"{BASE_URL}/v1/models"

    response = requests.get(url, timeout=10)
    response.raise_for_status()

    data = response.json()

    print(json.dumps(data, indent=2))

    return data


## 6. Benchmark Runner

In [ ]:
def run_benchmark(max_num_seqs):
    result_filename = f"max-num-seqs-{max_num_seqs}.json"

    result_path = RESULT_DIR / result_filename

    command = [
        "vllm",
        "bench",
        "serve",

        "--backend", "vllm",

        "--base-url", BASE_URL,

        "--model", MODEL,

        "--served-model-name", SERVED_NAME,

        "--dataset-name", "random",

        "--random-input-len", str(INPUT_LEN),

        "--random-output-len", str(OUTPUT_LEN),

        "--num-prompts", str(NUM_PROMPTS),

        "--request-rate", str(REQUEST_RATE),

        "--save-result",

        "--result-dir", str(RESULT_DIR),

        "--result-filename", result_filename,
    ]

    print("=" * 70)
    print(f"Running benchmark: max-num-seqs={max_num_seqs}")
    print("=" * 70)
    print(" ".join(command))
    print()

    result = subprocess.run(
        command,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"Benchmark failed for max-num-seqs={max_num_seqs}"
        )

    if not result_path.exists():
        raise FileNotFoundError(
            f"Expected result file not found: {result_path}"
        )

    print()
    print("Result saved:", result_path)

    return result_path


## 7. Smoke Test

In [ ]:
# Start with max-num-seqs=4 for the smoke test.
start_vllm(4)

wait_for_server()

get_model_info()


In [ ]:
# Simple OpenAI-compatible API test

payload = {
    "model": SERVED_NAME,
    "prompt": "Explain Kubernetes GPU scheduling in simple terms.",
    "max_tokens": 20,
    "temperature": 0
}

response = requests.post(
    f"{BASE_URL}/v1/completions",
    json=payload,
    timeout=120
)

print("HTTP status:", response.status_code)

response.raise_for_status()

data = response.json()

print(json.dumps(data, indent=2)[:3000])


## 8. Stop Smoke-Test Server

In [ ]:
def stop_vllm():
    global server_process

    if server_process is None:
        print("No server process.")
        return

    if server_process.poll() is None:
        print("Stopping vLLM server...")
        server_process.terminate()

        try:
            server_process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            print("Server did not stop gracefully. Killing...")
            server_process.kill()
            server_process.wait()

    else:
        print("vLLM server already stopped.")

    server_process = None


stop_vllm()


## 9. Run max-num-seqs Experiment

In [ ]:
benchmark_files = []

for max_num_seqs in MAX_NUM_SEQS_VALUES:

    print()
    print("#" * 80)
    print(f"# EXPERIMENT: max-num-seqs={max_num_seqs}")
    print("#" * 80)

    # Start fresh server for each configuration.
    start_vllm(max_num_seqs)

    try:
        wait_for_server()

        get_model_info()

        result_path = run_benchmark(max_num_seqs)

        benchmark_files.append(result_path)

    finally:
        stop_vllm()

    # Small pause between configurations.
    time.sleep(5)


print()
print("All benchmark runs completed.")
print("Files:")
for path in benchmark_files:
    print(" -", path)


## 10. Inspect Raw Results

In [ ]:
result_files = sorted(RESULT_DIR.glob("max-num-seqs-*.json"))

print("Raw result files:")

for path in result_files:
    print(path)


In [ ]:
if result_files:
    sample_file = result_files[0]

    with open(sample_file) as f:
        sample_result = json.load(f)

    print(json.dumps(sample_result, indent=2)[:10000])
else:
    print("No benchmark result files found.")


## 11. Parse Benchmark Results

In [ ]:
def find_metric(data, possible_names):
    for name in possible_names:
        if name in data:
            return data[name]

    return None


def parse_result(path):
    with open(path) as f:
        data = json.load(f)

    max_num_seqs = int(
        path.stem.replace("max-num-seqs-", "")
    )

    row = {
        "max_num_seqs": max_num_seqs,
        "file": str(path),
    }

    # Common vLLM benchmark metric names.
    row["request_throughput"] = find_metric(
        data,
        [
            "request_throughput",
            "request_throughput_req_s",
        ]
    )

    row["output_throughput"] = find_metric(
        data,
        [
            "output_throughput",
            "output_throughput_tok_s",
        ]
    )

    row["total_throughput"] = find_metric(
        data,
        [
            "total_token_throughput",
            "total_throughput",
        ]
    )

    row["mean_ttft_ms"] = find_metric(
        data,
        [
            "mean_ttft_ms",
            "mean_ttft",
        ]
    )

    row["median_ttft_ms"] = find_metric(
        data,
        [
            "median_ttft_ms",
            "median_ttft",
        ]
    )

    row["p95_ttft_ms"] = find_metric(
        data,
        [
            "p95_ttft_ms",
            "p95_ttft",
        ]
    )

    row["mean_tpot_ms"] = find_metric(
        data,
        [
            "mean_tpot_ms",
            "mean_tpot",
        ]
    )

    row["median_tpot_ms"] = find_metric(
        data,
        [
            "median_tpot_ms",
            "median_tpot",
        ]
    )

    row["p95_tpot_ms"] = find_metric(
        data,
        [
            "p95_tpot_ms",
            "p95_tpot",
        ]
    )

    row["completed"] = find_metric(
        data,
        [
            "completed",
            "completed_requests",
        ]
    )

    row["failed"] = find_metric(
        data,
        [
            "failed",
            "failed_requests",
        ]
    )

    return row


rows = []

for path in result_files:
    try:
        rows.append(parse_result(path))
    except Exception as e:
        print("Could not parse:", path)
        print("Reason:", e)


results_df = pd.DataFrame(rows)

results_df


## 12. Save Summary

In [ ]:
summary_path = RESULT_DIR / "benchmark-summary.csv"

results_df.to_csv(
    summary_path,
    index=False
)

print("Saved:", summary_path)
display(results_df)


## 13. Throughput vs max-num-seqs

In [ ]:
if "output_throughput" in results_df.columns:
    plot_df = results_df.dropna(
        subset=["output_throughput"]
    )

    if not plot_df.empty:
        plt.figure(figsize=(8, 5))

        plt.plot(
            plot_df["max_num_seqs"],
            plot_df["output_throughput"],
            marker="o"
        )

        plt.xlabel("max-num-seqs")
        plt.ylabel("Output throughput (tokens/s)")
        plt.title("Output Throughput vs max-num-seqs")
        plt.grid(True, alpha=0.3)

        plt.tight_layout()

        path = PLOT_DIR / "throughput-vs-max-num-seqs.png"
        plt.savefig(path, dpi=150)
        plt.show()

        print("Saved:", path)


## 14. p95 TTFT vs max-num-seqs

In [ ]:
if "p95_ttft_ms" in results_df.columns:
    plot_df = results_df.dropna(
        subset=["p95_ttft_ms"]
    )

    if not plot_df.empty:
        plt.figure(figsize=(8, 5))

        plt.plot(
            plot_df["max_num_seqs"],
            plot_df["p95_ttft_ms"],
            marker="o"
        )

        plt.xlabel("max-num-seqs")
        plt.ylabel("p95 TTFT (ms)")
        plt.title("p95 TTFT vs max-num-seqs")
        plt.grid(True, alpha=0.3)

        plt.tight_layout()

        path = PLOT_DIR / "p95-ttft-vs-max-num-seqs.png"
        plt.savefig(path, dpi=150)
        plt.show()

        print("Saved:", path)


## 15. p95 TPOT vs max-num-seqs

In [ ]:
if "p95_tpot_ms" in results_df.columns:
    plot_df = results_df.dropna(
        subset=["p95_tpot_ms"]
    )

    if not plot_df.empty:
        plt.figure(figsize=(8, 5))

        plt.plot(
            plot_df["max_num_seqs"],
            plot_df["p95_tpot_ms"],
            marker="o"
        )

        plt.xlabel("max-num-seqs")
        plt.ylabel("p95 TPOT (ms)")
        plt.title("p95 TPOT vs max-num-seqs")
        plt.grid(True, alpha=0.3)

        plt.tight_layout()

        path = PLOT_DIR / "p95-tpot-vs-max-num-seqs.png"
        plt.savefig(path, dpi=150)
        plt.show()

        print("Saved:", path)


## 16. Benchmark Summary

The experiment changes only `max-num-seqs` while keeping the model,
GPU, model length, memory utilization, request rate, input length,
and output length fixed.

The results should be used to study the trade-off between:

- throughput
- TTFT
- TPOT
- concurrency
- GPU utilization
- GPU memory pressure
- request latency
